# Message Intelligence Pipeline
### Classification - Task/Event Extraction - Sensitive Info Detection - Embedding Validation


## Setup & Data Load

In [ ]:
import pandas as pd
import re
import json
from datetime import datetime

IN_PATH = "messages.csv"
MANDATORY_PATH = "mandatory_demo_ids.csv"

df = pd.read_csv(IN_PATH)
df.columns = [c.strip() for c in df.columns]
df["timestamp"] = pd.to_datetime(df["timestamp"]) #-------> Converting str into date/time obj 
df = df.sort_values("timestamp").reset_index(drop=True)  #---> process in chronological order, per assignment rules

mandatory_ids = pd.read_csv(MANDATORY_PATH)["message_id"].tolist()

print(f"Loaded {len(df)} messages, {len(mandatory_ids)} mandatory demo IDs")
df.head()


Loaded 900 messages, 15 mandatory demo IDs


,message_id,timestamp,sender,message
0,MSG_0001,2026-09-01 08:00:00,Meera,"For today: Calendar update: family dinner, 202..."
1,MSG_0002,2026-09-01 08:37:00,Ishaan,Can you review the privacy checklist before 20...
2,MSG_0003,2026-09-01 09:14:00,Kabir,FYI: Reminder: mentor catch-up happens on 2026...
3,MSG_0004,2026-09-01 09:51:00,Aarav,One more thing: The training material is on th...
4,MSG_0005,2026-09-01 10:28:00,Aarav,"Hi, My home address is 42 Lake View Road, Chen..."


## Part 3: Sensitive Info Detection

Sensitive detection runs **before classification** so that a sensitive message can never be mis-filed into another category, and its raw value is masked before it can appear anywhere downstream (logs, other outputs, this notebook's own printed cells).


In [ ]:
DATE_RE = r"\d{4}-\d{2}-\d{2}"

SENSITIVE_PATTERNS = [
    #-----> (type, compiled regex with one capture group around the sensitive value, risk level)
    ("password", re.compile(r"password\s+([A-Za-z0-9#\-]+)", re.I), "high"),
    ("one_time_password", re.compile(r"OTP\s+is\s+([0-9\-]+)", re.I), "high"),
    ("account_recovery_code", re.compile(r"recovery code is\s+([A-Za-z0-9\-]+)", re.I), "high"),
    ("card_number", re.compile(r"card number is\s+([0-9 \-]+)", re.I), "high"),
    ("auth_token", re.compile(r"access token is\s+([A-Za-z0-9_\-]+)", re.I), "high"),
    ("home_address", re.compile(r"home address is\s+(.+?)(?:\.|$)", re.I), "medium"),
    ("identification_number", re.compile(r"identification number is\s+([A-Za-z0-9\-]+)", re.I), "high"),
    ("phone_number", re.compile(r"contact me on\s+([\d \-]+)", re.I), "medium"),
    ("bank_account_number", re.compile(r"bank account number\s+([\d\-]+)", re.I), "high"),
]

def mask_value(val):
    """Returns first 2 and last 2 characters, mask the rest. Values <=4 chars are fully masked."""
    val = val.strip().rstrip(".")
    if len(val) <= 4:
        return "*" * len(val)
    return val[:2] + "*" * (len(val) - 4) + val[-2:]

def detect_sensitive(msg):
    for stype, pattern, risk in SENSITIVE_PATTERNS:
        m = pattern.search(msg)
        if m:
            raw = m.group(1)
            masked_val = mask_value(raw)
            masked_text = msg[:m.start(1)] + masked_val + msg[m.end(1):]
            action = "do_not_store" if risk == "high" else "ask_for_confirmation"
            return {
                "sensitivity_type": stype,
                "risk": risk,
                "masked_text": masked_text,
                "recommended_action": action,
            }
    return None

#------> Quick Test:
print(detect_sensitive("One more thing: My card number is 4111 1111 1111 1111-92."))
print(detect_sensitive("Just checking\u2014Your OTP is 482193-50. It expires in 10 minutes."))


{'sensitivity_type': 'card_number', 'risk': 'high', 'masked_text': 'One more thing: My card number is 41******************92.', 'recommended_action': 'do_not_store'}
{'sensitivity_type': 'one_time_password', 'risk': 'high', 'masked_text': 'Just checking—Your OTP is 48*****50. It expires in 10 minutes.', 'recommended_action': 'do_not_store'}


## Part 1: Message Classification

Messages are checked against **ordered** pattern groups — first match wins. The order matters because a message can superficially match more than one category (e.g. a promotional message containing "Please note:", which could otherwise be mistaken for an action-required phrase).

**Priority order**: `sensitive_information` -> `promotional` -> `meeting_or_event` -> `action_required` -> `personal_information` -> `general_information` (fallback).

In [3]:
PROMO_PATTERNS = [
    (re.compile(r"\bUse code SAVE\d+\b", re.I), "promo code present"),
    (re.compile(r"\b(discount|off selected|flash sale|subscription|premium plan|reward points|coupon|save \d+%)\b", re.I),
     "promotional/marketing language"),
]

MEETING_PATTERNS = [
    (re.compile(r"\bCalendar update\b", re.I), "explicit calendar update phrasing"),
    (re.compile(r"\bhappens on\s+" + DATE_RE, re.I), "'happens on <date>' meeting/reminder phrasing"),
    (re.compile(r"\bscheduled for\s+" + DATE_RE, re.I), "'scheduled for <date>' phrasing"),
    (re.compile(r"\bjoin the .* on\s+" + DATE_RE, re.I), "invite to join an event on a date"),
    (re.compile(r"\bavailable for the .* at\b", re.I), "availability request for an event"),
]

ACTION_PATTERNS = [
    (re.compile(r"\bdeadline is\s+" + DATE_RE, re.I), "explicit deadline stated"),
    (re.compile(r"\bis due on\s+" + DATE_RE, re.I), "explicit due date stated"),
    (re.compile(r"\b(review|reply|confirm|complete|upload|submit|renew|pay|email|update|book|send|finish)\b.*\b(by|before)\s+" + DATE_RE, re.I),
     "action verb requested with a date constraint ('by'/'before')"),
    (re.compile(r"^(Can you|Could you) (review|update|confirm|send|call|finish)\b", re.I), "direct question requesting an action"),
    (re.compile(r"\bI need you to\b", re.I), "explicit request phrasing"),
    (re.compile(r"\bDon'?t forget to\b", re.I), "explicit reminder-to-act phrasing"),
    (re.compile(r"\bPlease call\b", re.I), "explicit request to call someone"),
]

PERSONAL_PATTERNS = [
    (re.compile(r"\bPersonal note\b", re.I), "explicit 'Personal note' tag"),
    (re.compile(r"\bmy (favourite|emergency contact|home address|identification number)\b", re.I),
     "first-person personal detail disclosed"),
    (re.compile(r"\bi (am|drink|use|prefer)\b", re.I), "first-person preference/trait statement"),
    (re.compile(r"\btest result\b", re.I), "personal health information"),
]

def classify(msg):
    sens = detect_sensitive(msg)
    if sens:
        stype = sens["sensitivity_type"]
        return "sensitive_information", 0.97, f"Matches {stype} pattern; flagged before general classification for safety.", sens

    for pattern, reason in PROMO_PATTERNS:
        if pattern.search(msg):
            return "promotional", 0.90, reason, None

    for pattern, reason in MEETING_PATTERNS:
        if pattern.search(msg):
            return "meeting_or_event", 0.88, reason, None

    for pattern, reason in ACTION_PATTERNS:
        if pattern.search(msg):
            return "action_required", 0.87, reason, None

    for pattern, reason in PERSONAL_PATTERNS:
        if pattern.search(msg):
            return "personal_information", 0.85, reason, None

    return "general_information", 0.60, "No task, event, personal, promotional, or sensitive pattern matched; treated as informational default.", None


In [4]:
classification_results = []
sensitive_results = []

for _, row in df.iterrows():
    msg = row["message"]
    category, confidence, reason, sens_info = classify(msg)
    classification_results.append({
        "message_id": row["message_id"],
        "category": category,
        "confidence": confidence,
        "reason": reason,
    })
    if sens_info:
        sensitive_results.append({"message_id": row["message_id"], **sens_info})

class_df = pd.DataFrame(classification_results)
print(class_df["category"].value_counts())
class_df.head(10)


category
general_information      259
action_required          211
meeting_or_event         150
promotional              100
sensitive_information     90
personal_information      90
Name: count, dtype: int64


,message_id,category,confidence,reason
0,MSG_0001,meeting_or_event,0.88,explicit calendar update phrasing
1,MSG_0002,action_required,0.87,action verb requested with a date constraint (...
2,MSG_0003,meeting_or_event,0.88,'happens on <date>' meeting/reminder phrasing
3,MSG_0004,general_information,0.60,"No task, event, personal, promotional, or sens..."
4,MSG_0005,sensitive_information,0.97,Matches home_address pattern; flagged before g...
5,MSG_0006,general_information,0.60,"No task, event, personal, promotional, or sens..."
6,MSG_0007,action_required,0.87,action verb requested with a date constraint (...
7,MSG_0008,general_information,0.60,"No task, event, personal, promotional, or sens..."
8,MSG_0009,personal_information,0.85,first-person personal detail disclosed
9,MSG_0010,action_required,0.87,explicit deadline stated


## Part 2: Task & Event Extraction

Only runs on messages already classified `action_required` or `meeting_or_event`. **Nothing is guessed** - if a field (date, time, person) can't be confidently resolved from the text, it's stored as the literal string `"unresolved"`, per the assignment's explicit instruction not to invent missing information.

In [5]:
def extract_date(msg):
    m = re.search(DATE_RE, msg)
    return m.group(0) if m else None

def extract_time(msg):
    m = re.search(r"\b(\d{1,2}:\d{2})\b", msg)
    if m:
        return m.group(1)
    m2 = re.search(r"\b(\d{1,2}\s?(AM|PM))\b", msg, re.I)
    if m2:
        return m2.group(1)
    return None

def extract_person(msg, sender):
    # Only attribute a person if the message explicitly references one via "with <Name>".
    # We deliberately do NOT default to the sender, since that would be inventing an
    # assumption the message text doesn't support.
    m = re.search(r"\bwith\s+([A-Z][a-z]+)\b", msg)
    if m:
        return m.group(1)
    return None

def extract_title(msg):
    cleaned = re.sub(r"^(Hi,|FYI:|Please note:|Just checking\u2014|Can you help\?|"
                      r"For today:|One more thing:|Quick update:|Important:)\s*",
                      "", msg).strip()
    cleaned = re.split(r"\s*(?:before|by|deadline is|is due on|happens on|scheduled for)\b",
                        cleaned, maxsplit=1)[0].strip(" .,;:?")
    cleaned = re.sub(r"^(Can you|Could you|Don't forget to|I need you to|Please)\s+", "", cleaned, flags=re.I)
    cleaned = cleaned[0].upper() + cleaned[1:] if cleaned else cleaned
    return cleaned if cleaned else msg.strip()

def extract_task_or_event(row, category):
    msg = row["message"]
    date = extract_date(msg)
    time_ = extract_time(msg)
    person = extract_person(msg, row["sender"])
    title = extract_title(msg)

    if category == "action_required":
        item_type = "task"
        priority = "high" if date else "medium"
    else:
        item_type = "event"
        priority = "medium"

    return {
        "type": item_type,
        "title": title if title else "unresolved",
        "description": msg,
        "date_or_deadline": date if date else "unresolved",
        "time": time_ if time_ else "unresolved",
        "person": person if person else "unresolved",
        "priority": priority,
        "source_message_id": row["message_id"],
    }


In [6]:
extraction_results = []
task_counter = 1
merged_for_extraction = df.merge(class_df, on="message_id")

for _, row in merged_for_extraction.iterrows():
    if row["category"] in ("action_required", "meeting_or_event"):
        item = extract_task_or_event(row, row["category"])
        item_type_label = "TASK" if item["type"] == "task" else "EVENT"
        item["item_id"] = f"{item_type_label}_{task_counter:04d}"
        task_counter += 1
        extraction_results.append(item)

extraction_df = pd.DataFrame(extraction_results)
n_tasks = (extraction_df["type"] == "task").sum()
n_events = (extraction_df["type"] == "event").sum()
print(f"Extracted {len(extraction_df)} items: {n_tasks} tasks, {n_events} events")
extraction_df.head(5)


Extracted 361 items: 211 tasks, 150 events


,type,title,description,date_or_deadline,time,person,priority,source_message_id,item_id
0,event,"Calendar update: family dinner, 2026-09-19 at ...","For today: Calendar update: family dinner, 202...",2026-09-19,10:00,unresolved,medium,MSG_0001,EVENT_0001
1,task,Review the privacy checklist,Can you review the privacy checklist before 20...,2026-09-09,unresolved,unresolved,high,MSG_0002,TASK_0002
2,event,Reminder: mentor catch-up,FYI: Reminder: mentor catch-up happens on 2026...,2026-09-16,11:00,unresolved,medium,MSG_0003,EVENT_0003
3,task,Reply to the client email,For today: Please reply to the client email by...,2026-09-04,unresolved,unresolved,high,MSG_0007,TASK_0004
4,task,Pay the electricity bill,Can you help? Don't forget to pay the electric...,2026-09-09,unresolved,unresolved,high,MSG_0010,TASK_0005




- Checking for at least one extracted item should have an `unresolved` field, ---> as mentioned in requirements! 

In [7]:
unresolved_examples = extraction_df[
    (extraction_df["time"] == "unresolved") |
    (extraction_df["person"] == "unresolved") |
    (extraction_df["date_or_deadline"] == "unresolved")
]
print(f"{len(unresolved_examples)} extracted items have at least one unresolved field")
unresolved_examples.head(5)


361 extracted items have at least one unresolved field


,type,title,description,date_or_deadline,time,person,priority,source_message_id,item_id
0,event,"Calendar update: family dinner, 2026-09-19 at ...","For today: Calendar update: family dinner, 202...",2026-09-19,10:00,unresolved,medium,MSG_0001,EVENT_0001
1,task,Review the privacy checklist,Can you review the privacy checklist before 20...,2026-09-09,unresolved,unresolved,high,MSG_0002,TASK_0002
2,event,Reminder: mentor catch-up,FYI: Reminder: mentor catch-up happens on 2026...,2026-09-16,11:00,unresolved,medium,MSG_0003,EVENT_0003
3,task,Reply to the client email,For today: Please reply to the client email by...,2026-09-04,unresolved,unresolved,high,MSG_0007,TASK_0004
4,task,Pay the electricity bill,Can you help? Don't forget to pay the electric...,2026-09-09,unresolved,unresolved,high,MSG_0010,TASK_0005


## Part 4: Embedding-Based Validation (ML component)

The core classifier above is rule-based. This section adds an **independent, embedding-based cross-check** using TF-IDF vectors + cosine similarity — this runs entirely locally with scikit-learn (no external API calls)

**Method**:
1. Vectorize all 900 messages with TF-IDF (1–2 grams).
2. Take the rule-based classifier's **highest-confidence** hits (≥ 0.85) per category as "anchor" examples.
3. For every message, find its nearest anchor by cosine similarity.
4. Compare the nearest-anchor category to the rule-based category 

This is a **validation signal on top of the rule engine**, not a replacement classifier — it's a lightweight way to sanity-check the rules against a genuinely independent (unsupervised, non-regex) method.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(merged_for_extraction["message"])

HIGH_CONF_THRESHOLD = 0.85
anchors = merged_for_extraction[merged_for_extraction["confidence"] >= HIGH_CONF_THRESHOLD].groupby("category").head(15)
anchor_indices = anchors.index.to_numpy()
anchor_categories = anchors["category"].to_numpy()
anchor_vectors = tfidf_matrix[anchor_indices]

print("Anchor set sizes (per category, from high-confidence rule hits):")
print(anchors["category"].value_counts())


Anchor set sizes (per category, from high-confidence rule hits):
category
meeting_or_event         15
action_required          15
sensitive_information    15
personal_information     15
promotional              15
Name: count, dtype: int64


In [9]:
similarities = cosine_similarity(tfidf_matrix, anchor_vectors)
nearest_anchor_idx = similarities.argmax(axis=1)
nearest_sim_score = similarities.max(axis=1)
nearest_category = anchor_categories[nearest_anchor_idx]

merged_for_extraction["embedding_nearest_category"] = nearest_category
merged_for_extraction["embedding_similarity"] = nearest_sim_score.round(3)
merged_for_extraction["rule_embedding_agree"] = (
    merged_for_extraction["category"] == merged_for_extraction["embedding_nearest_category"]
)

agreement_rate = merged_for_extraction["rule_embedding_agree"].mean()
print(f"Rule-based vs embedding-nearest agreement: {agreement_rate:.1%}")


Rule-based vs embedding-nearest agreement: 67.1%


**Note on the ~67% agreement rate**: this is expected, as `general_information` has zero anchors (its rule-based confidence is fixed at 0.60, below the 0.85 threshold, since it's a fallback with no distinguishing pattern of its own) — so every `general_information` message gets force-matched to *some* other category's nearest anchor, which mechanically produces disagreements. 

**Solution**(Below code) - Filtering these out... shows much stronger agreement on the categories that do have anchors.

In [ ]:
#----> Agreement rate excluding the general_information fallback (which has no anchors by design)
non_general = merged_for_extraction[merged_for_extraction["category"] != "general_information"]
non_general_agreement = non_general["rule_embedding_agree"].mean()
print(f"Agreement rate on non-general_information messages: {non_general_agreement:.1%}")

#-----> Most interesting disagreements: high similarity score but different category
disagreements = merged_for_extraction[~merged_for_extraction["rule_embedding_agree"]].sort_values(
    "embedding_similarity", ascending=False
)
disagreements[["message_id", "message", "category", "embedding_nearest_category", "embedding_similarity"]].head(10)


Agreement rate on non-general_information messages: 94.2%


,message_id,message,category,embedding_nearest_category,embedding_similarity
463,MSG_0464,Please note: Maya asked whether the demo was r...,general_information,action_required,0.441
369,MSG_0370,Quick update: The project folder was reorganized.,general_information,action_required,0.355
238,MSG_0239,"For today: For my profile, i live near the cen...",general_information,personal_information,0.355
440,MSG_0441,"Can you help? For my profile, i usually study ...",general_information,personal_information,0.307
391,MSG_0392,Can you help? The project folder was reorganized.,general_information,meeting_or_event,0.276
428,MSG_0429,"FYI: For my profile, i live near the central l...",general_information,personal_information,0.268
400,MSG_0401,I need you to back up the project files by 202...,action_required,meeting_or_event,0.243
863,MSG_0864,One more thing: I need you to back up the proj...,action_required,meeting_or_event,0.218
818,MSG_0819,Please note: Share the meeting notes is due on...,action_required,meeting_or_event,0.212
850,MSG_0851,Please note: Prepare the demo video is due on ...,action_required,meeting_or_event,0.210


## Part 5: Mandatory Demo IDs & Summary

In [11]:
mandatory_subset = class_df[class_df["message_id"].isin(mandatory_ids)]
print(f"Mandatory IDs covered: {len(mandatory_subset)} / {len(mandatory_ids)}")
mandatory_subset


Mandatory IDs covered: 15 / 15


,message_id,category,confidence,reason
0,MSG_0001,meeting_or_event,0.88,explicit calendar update phrasing
1,MSG_0002,action_required,0.87,action verb requested with a date constraint (...
2,MSG_0003,meeting_or_event,0.88,'happens on <date>' meeting/reminder phrasing
3,MSG_0004,general_information,0.60,"No task, event, personal, promotional, or sens..."
4,MSG_0005,sensitive_information,0.97,Matches home_address pattern; flagged before g...
5,MSG_0006,general_information,0.60,"No task, event, personal, promotional, or sens..."
6,MSG_0007,action_required,0.87,action verb requested with a date constraint (...
8,MSG_0009,personal_information,0.85,first-person personal detail disclosed
11,MSG_0012,general_information,0.60,"No task, event, personal, promotional, or sens..."
12,MSG_0013,sensitive_information,0.97,Matches card_number pattern; flagged before ge...


In [12]:
summary = {
    "total_messages": len(df),
    "category_counts": class_df["category"].value_counts().to_dict(),
    "total_tasks_extracted": int((extraction_df["type"] == "task").sum()),
    "total_events_extracted": int((extraction_df["type"] == "event").sum()),
    "total_sensitive_flagged": len(sensitive_results),
    "mandatory_ids_covered": len(mandatory_subset),
    "embedding_validation_agreement_rate": round(float(agreement_rate), 4),
}
print(json.dumps(summary, indent=2))


{
  "total_messages": 900,
  "category_counts": {
    "general_information": 259,
    "action_required": 211,
    "meeting_or_event": 150,
    "promotional": 100,
    "sensitive_information": 90,
    "personal_information": 90
  },
  "total_tasks_extracted": 211,
  "total_events_extracted": 150,
  "total_sensitive_flagged": 90,
  "mandatory_ids_covered": 15,
  "embedding_validation_agreement_rate": 0.6711
}


## Save structured output files

These JSON files are the "generated structured output files" deliverable required by the assignment.

In [13]:
import os
os.makedirs("outputs", exist_ok=True)

with open("outputs/classification_results.json", "w") as f:
    json.dump(classification_results, f, indent=2)

with open("outputs/extraction_results.json", "w") as f:
    json.dump(extraction_results, f, indent=2)

with open("outputs/sensitive_detection_results.json", "w") as f:
    json.dump(sensitive_results, f, indent=2)

with open("outputs/mandatory_ids_results.json", "w") as f:
    json.dump(mandatory_subset.to_dict(orient="records"), f, indent=2)

with open("outputs/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved all output files to ./outputs/")


Saved all output files to ./outputs/


## Example - One classification decision

- Walking through `MSG_0001` end to end as an example: 
- To check whether my code classify and extract the given msg

In [14]:
example_id = "MSG_0001"
example_msg = df[df["message_id"] == example_id]["message"].values[0]
example_result = [r for r in classification_results if r["message_id"] == example_id][0]

print("Message:", example_msg)
print("\nClassification:", json.dumps(example_result, indent=2))

example_extraction = [e for e in extraction_results if e["source_message_id"] == example_id]
if example_extraction:
    print("\nExtracted item:", json.dumps(example_extraction[0], indent=2))


Message: For today: Calendar update: family dinner, 2026-09-19 at 10:00, the library.

Classification: {
  "message_id": "MSG_0001",
  "category": "meeting_or_event",
  "confidence": 0.88,
  "reason": "explicit calendar update phrasing"
}

Extracted item: {
  "type": "event",
  "title": "Calendar update: family dinner, 2026-09-19 at 10:00, the library",
  "description": "For today: Calendar update: family dinner, 2026-09-19 at 10:00, the library.",
  "date_or_deadline": "2026-09-19",
  "time": "10:00",
  "person": "unresolved",
  "priority": "medium",
  "source_message_id": "MSG_0001",
  "item_id": "EVENT_0001"
}
